# Keep ratio testing

Testing how many data points are required for certain metrics to be consistent

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import plotly.graph_objects as go

In [ ]:
import wavelet_transform.utils.statistical_features as sf
from wavelet_transform.utils.preprocessing import preprocess

Make signals

In [ ]:
sampling_rate = 200e6
frequency = 1.1e6 * 2

# Define test lengths as powers of base
base = 2
min_power = 2  # base^min_len data points
max_len = 1e6
max_power = int(np.emath.logn(base, max_len))  # base^max_len data points

t = np.arange(0, (base**max_power)/sampling_rate, 1 / sampling_rate)
sine_wave = np.sin(2 * np.pi * frequency * t)  # 5 MHz sine wave
noise = np.random.normal(0, 1, sine_wave.shape) * np.std(sine_wave) * 0.1
signal = sine_wave + noise
test_signals = [sine_wave, noise, signal]
test_signal_names = ["Sine wave", "Gaussian noise", "Sine wave + noise"]

Plot signal

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x = t,
        y = signal,
        mode = "lines",
        name = "Signal"
    )
)
fig.show()

Trim signal to nearest full cycle

In [ ]:
def len_full_cycles(sampling_rate, frequency, length):
    ppw = sampling_rate / frequency
    new_len = int(length / ppw) * ppw
    new_len = max(new_len, ppw)
    return new_len

In [ ]:
power_lengths = [base ** i for i in range(min_power, max_power + 1)]
statistical_metrics = [
    sf.kurtosis,
    # sf.log_avg_energy,
    sf.avg_energy,
    sf.spectral_entropy,
    # sf.crest_factor,
]
trimmed_lengths = [len_full_cycles(sampling_rate, frequency, length) for length in power_lengths]

In [ ]:
signal_lengths = trimmed_lengths  # Use trimmed lengths to keep full cycles
signal_lengths = power_lengths  # Use power lengths directly
window = False

for signal, signal_name in zip(test_signals, test_signal_names):
    results = []
    for length in signal_lengths:
        sig_segment = signal[:int(length)]
        sig_segment = preprocess(sig_segment, window=window)
        metrics_values = []
        for metric in statistical_metrics:
            value = metric(sig_segment)
            metrics_values.append(value)
        metrics_values = np.array(metrics_values)
        results.append(metrics_values)

    fig = go.Figure()
    for i, metric in enumerate(statistical_metrics):
        r = [result[i] for result in results]
        r = np.array(r)
        fig.add_trace(
            go.Scatter(
                x=signal_lengths,
                y=r,
                mode="lines+markers",
                name=metric.__name__,
            ),
        )
        fig.update_layout(
            title=f"{signal_name}",
            xaxis_title="Signal Length (number of data points)",
            yaxis_title="Metric Value",
            xaxis_type="log"
        )
    fig.show()